In [1]:
%pip install pymupdf


[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
%pip install pypdf scikit-learn


[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [3]:
# 1. Imports and API client setup
import os
import re
import csv
import pandas as pd
from pypdf import PdfReader
from dotenv import load_dotenv
from openai import OpenAI
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Load environment file if present.
# Do NOT hard-code the API key in this notebook.
load_dotenv(dotenv_path="environment")
load_dotenv(dotenv_path="../environment")
load_dotenv(dotenv_path="../../environment")

api_key = os.getenv("CSCS_API_KEY", "").strip()

if not api_key:
    raise ValueError(
        "CSCS_API_KEY was not found. Set it in the environment file or run: "
        "os.environ['CSCS_API_KEY'] = 'your_key_here'"
    )

client_cscs = OpenAI(
    base_url="https://api.swissai.svc.cscs.ch/v1",
    api_key=api_key
)

MODEL_NAME = "moonshotai/Kimi-K2.5-SDSC"
print("API client ready.")
print("Using model:", MODEL_NAME)

API client ready.
Using model: moonshotai/Kimi-K2.5-SDSC


In [4]:
# 2. Locate PDFs and qa_set.csv robustly

def find_file(filename, search_roots):
    matches = []
    for search_root in search_roots:
        if os.path.exists(search_root):
            for root, dirs, files in os.walk(search_root):
                if filename in files:
                    matches.append(os.path.join(root, filename))
    return matches

def find_pdfs(search_roots):
    pdfs = []
    for search_root in search_roots:
        if os.path.exists(search_root):
            for root, dirs, files in os.walk(search_root):
                for file in files:
                    if file.lower().endswith(".pdf"):
                        path = os.path.join(root, file)
                        if path not in pdfs:
                            pdfs.append(path)
    return pdfs

search_roots = [
    ".",
    "..",
    "../..",
    "/home/renku/work/Durham-Hackathon-2026-template",
    "/workspace/source",
    "/home/renku/work"
]

pdf_paths = find_pdfs(search_roots)
qa_matches = find_file("qa_set.csv", search_roots)

if not pdf_paths:
    raise FileNotFoundError("No PDF files found. Check the dataset path.")
if not qa_matches:
    raise FileNotFoundError("qa_set.csv not found. Check the data/a2q folder.")

# Prefer the project data path if present.
qa_path = sorted(qa_matches, key=lambda x: ("data/a2q" not in x, len(x)))[0]

print("PDFs found:", len(pdf_paths))
for p in pdf_paths:
    print("-", p)

print("\nQA file:", qa_path)

PDFs found: 8
- ../../a2q-answer-any-question-dataset/Web Version _E-Government Survey 2024 11102024.pdf
- ../../a2q-answer-any-question-dataset/World_Inequality_Report_2026.pdf
- ../../a2q-answer-any-question-dataset/natural-catastrophe-and-climate-report-2023.pdf
- ../../a2q-answer-any-question-dataset/swissre_sigma-1_2024_english.pdf
- /home/renku/work/a2q-answer-any-question-dataset/Web Version _E-Government Survey 2024 11102024.pdf
- /home/renku/work/a2q-answer-any-question-dataset/World_Inequality_Report_2026.pdf
- /home/renku/work/a2q-answer-any-question-dataset/natural-catastrophe-and-climate-report-2023.pdf
- /home/renku/work/a2q-answer-any-question-dataset/swissre_sigma-1_2024_english.pdf

QA file: ../data/a2q/qa_set.csv


In [5]:
# 3. Load questions
qa_df = pd.read_csv(qa_path, sep=";").fillna("")

print("QA shape:", qa_df.shape)
print("Columns:", qa_df.columns.tolist())

if "question" not in qa_df.columns:
    raise ValueError("qa_set.csv must contain a 'question' column.")

qa_df.head(10)

QA shape: (40, 2)
Columns: ['question', 'answer']


,question,answer
0,What year where there most casualties from man...,"2002. In that year more than 10,000 casualties..."
1,In what year did the quantity of man-made disa...,Man-made disasters peaked in 2005
2,Between 1970 and 2023 what year in the data sh...,2023. There were 218 instances of natural cat...
3,What year between 1994 and 2023 had the most h...,2011 with 6.
4,How much higher are the 2023 insured losses th...,They are higher by 21%.
5,Which figure shows the trend in insured losses...,
6,"Before 2023, what was the highest year on reco...",
7,What regions does the Swiss Re report on natur...,
8,What is the highest Benefit to Cost ratio buil...,
9,According to the Swiss Re Institute report on ...,


In [6]:
# 4. Load PDF text - faster version using PyMuPDF

import os
import re
import fitz  # PyMuPDF
import pandas as pd

documents = []

for pdf_path in pdf_paths:
    print("Reading:", pdf_path)
    
    try:
        pdf_doc = fitz.open(pdf_path)
        
        for page_index in range(len(pdf_doc)):
            page = pdf_doc.load_page(page_index)
            text = page.get_text("text")
            
            if text and len(text.strip()) > 50:
                documents.append({
                    "source": os.path.basename(pdf_path),
                    "page": page_index + 1,
                    "text": re.sub(r"\s+", " ", text.strip())
                })
        
        pdf_doc.close()
        
    except Exception as e:
        print("Failed to read:", pdf_path, "|", e)

print("Total pages loaded:", len(documents))
print(pd.Series([d["source"] for d in documents]).value_counts())

if not documents:
    raise ValueError("No text was extracted from PDFs.")

Reading: ../../a2q-answer-any-question-dataset/Web Version _E-Government Survey 2024 11102024.pdf
Reading: ../../a2q-answer-any-question-dataset/World_Inequality_Report_2026.pdf
Reading: ../../a2q-answer-any-question-dataset/natural-catastrophe-and-climate-report-2023.pdf
Reading: ../../a2q-answer-any-question-dataset/swissre_sigma-1_2024_english.pdf
Reading: /home/renku/work/a2q-answer-any-question-dataset/Web Version _E-Government Survey 2024 11102024.pdf
Reading: /home/renku/work/a2q-answer-any-question-dataset/World_Inequality_Report_2026.pdf
Reading: /home/renku/work/a2q-answer-any-question-dataset/natural-catastrophe-and-climate-report-2023.pdf
Reading: /home/renku/work/a2q-answer-any-question-dataset/swissre_sigma-1_2024_english.pdf
Total pages loaded: 994
Web Version _E-Government Survey 2024 11102024.pdf    396
World_Inequality_Report_2026.pdf                      386
natural-catastrophe-and-climate-report-2023.pdf       138
swissre_sigma-1_2024_english.pdf                    

In [7]:
# 5. Split pages into chunks

def split_text(text, chunk_size=1200, overlap=250):
    chunks = []
    start = 0
    text = str(text)

    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end].strip()
        if chunk:
            chunks.append(chunk)
        start += chunk_size - overlap

    return chunks

chunks = []

for doc in documents:
    for chunk_id, chunk_text in enumerate(split_text(doc["text"], chunk_size=1200, overlap=250)):
        chunks.append({
            "source": doc["source"],
            "page": doc["page"],
            "chunk_id": chunk_id,
            "text": chunk_text
        })

print("Total chunks:", len(chunks))
print(pd.Series([c["source"] for c in chunks]).value_counts())

if not chunks:
    raise ValueError("No chunks were created.")

Total chunks: 3376
Web Version _E-Government Survey 2024 11102024.pdf    1548
World_Inequality_Report_2026.pdf                      1148
natural-catastrophe-and-climate-report-2023.pdf        430
swissre_sigma-1_2024_english.pdf                       250
Name: count, dtype: int64


In [8]:
# 6. Build TF-IDF retriever
chunk_texts = [c["text"] for c in chunks]

vectorizer = TfidfVectorizer(
    stop_words="english",
    max_features=50000,
    ngram_range=(1, 2)
)

tfidf_matrix = vectorizer.fit_transform(chunk_texts)

print("TF-IDF matrix shape:", tfidf_matrix.shape)

TF-IDF matrix shape: (3376, 50000)


In [10]:
def choose_sources_by_question(question):
    q = str(question).lower()

    # Swiss Re / natural catastrophe / climate disasters
    if any(k in q for k in [
        "man-made disaster", "man-made disasters",
        "natural catastrophe", "natural catastrophes",
        "insured loss", "insured losses",
        "uninsured", "casualties",
        "severe convective storm",
        "catastrophe", "catastrophes",
        "disaster", "disasters",
        "latin america",
        "environmental disasters",
        "costliest",
        "benefit-cost ratio",
        "building code",
        "weather events"
    ]):
        return ["swissre", "natural-catastrophe", "climate"]

    # E-government / EGDI / OSI
    if any(k in q for k in [
        "egdi", "e-government", "online services",
        "osi", "telecommunications infrastructure",
        "web-portals", "web portals",
        "member states", "un 193",
        "income tax online",
        "death certificate",
        "vehicle registration",
        "digital invoicing",
        "e-procurement",
        "mobile", "app",
        "landlocked",
        "small island",
        "gdi group",
        "government digital service"
    ]):
        return ["e-government", "egdi", "survey"]

    # World Inequality / energy
    if any(k in q for k in [
        "inequality", "energy transition",
        "fossil fuel", "power generation",
        "investment segments"
    ]):
        return ["world_inequality", "inequality"]

    # fallback: search all PDFs
    return []

In [11]:
from sklearn.metrics.pairwise import cosine_similarity

def retrieve_relevant_chunks_filtered(question, top_k=20):
    question_vec = vectorizer.transform([question])
    similarities = cosine_similarity(question_vec, tfidf_matrix).flatten()

    source_keywords = choose_sources_by_question(question)

    candidate_indices = []

    if source_keywords:
        for idx, chunk in enumerate(chunks):
            source_name = chunk["source"].lower()
            if any(keyword in source_name for keyword in source_keywords):
                candidate_indices.append(idx)

    if len(candidate_indices) == 0:
        candidate_indices = list(range(len(chunks)))

    ranked = sorted(
        candidate_indices,
        key=lambda idx: similarities[idx],
        reverse=True
    )[:top_k]

    results = []
    for idx in ranked:
        item = chunks[idx].copy()
        item["score"] = similarities[idx]
        results.append(item)

    return results

In [12]:
def rerank_evidence(question, evidence):
    q = question.lower()
    reranked = []

    for e in evidence:
        text = e["text"].lower()
        score = e["score"]
        bonus = 0.0


        if "figure" in text or "table" in text or "chart" in text:
            bonus += 0.08


        if any(w in q for w in ["year", "highest", "most", "percentage", "how many", "increase"]):
            digit_count = sum(c.isdigit() for c in text)
            if digit_count > 20:
                bonus += 0.08


        if any(w in q for w in ["casualties", "victims", "deaths"]):
            if any(w in text for w in ["victims", "casualties", "fatalities", "deaths"]):
                bonus += 0.12


        if "insured losses" in q and "insured losses" in text:
            bonus += 0.12


        if any(w in q for w in ["egdi", "osi", "tii", "online services", "telecommunications infrastructure"]):
            if any(w in text for w in ["egdi", "osi", "tii", "online service", "telecommunications infrastructure"]):
                bonus += 0.12


        if "definition" in text or "definitions" in text:
            bonus -= 0.05

        e2 = e.copy()
        e2["rerank_score"] = score + bonus
        reranked.append(e2)

    reranked.sort(key=lambda x: x["rerank_score"], reverse=True)
    return reranked

In [13]:
MODEL_CANDIDATES = [
    "moonshotai/Kimi-K2.5-SDSC",
    "zai-org/GLM-4.7-Flash",
    "swiss-ai/Apertus-8B-Instruct-2509",
    "swiss-ai/Apertus-70B-Instruct-2509"
]


def deduplicate_evidence(evidence):
    seen = set()
    unique = []

    for e in evidence:
        key = (e["source"], e["page"], e.get("chunk_id", None))
        if key not in seen:
            unique.append(e)
            seen.add(key)

    return unique


def call_model_text(prompt, model):
    response = client_cscs.chat.completions.create(
        model=model,
        messages=[
            {"role": "user", "content": prompt}
        ],
        max_tokens=300,
        temperature=0
    )

    content = response.choices[0].message.content

    if content is None:
        return ""

    return str(content).strip()


def answer_question_text_only(question, top_k=10):
    evidence = retrieve_relevant_chunks_filtered(question, top_k=top_k)
    evidence = rerank_evidence(question, evidence)
    evidence = deduplicate_evidence(evidence)

    context_parts = []
    for e in evidence[:3]:
        context_parts.append(
            f"Source: {e['source']}, page {e['page']}\n{e['text'][:700]}"
        )

    context = "\n\n".join(context_parts)

    prompt = f"""
Answer the question using the evidence.

Question:
{question}

Evidence:
{context}

Rules:
- Give only the final answer.
- Keep it short.
- If the answer is a year, number, percentage, country, region, organisation, or figure number, give the exact value first.
- If the evidence strongly suggests an answer, give the best answer.
- Do not explain your reasoning.
- Do not say you cannot answer unless the evidence is completely unrelated.
"""

    for model in MODEL_CANDIDATES:
        try:
            answer = call_model_text(prompt, model)

            if answer and answer.strip() and "chatcompletion" not in answer.lower():
                return answer.strip(), evidence, model

        except Exception as e:
            print("Model failed:", model, "|", e)

    return "I cannot find enough evidence.", evidence, ""

In [14]:
results = []

for i, row in qa_df.iterrows():
    question = row["question"]
    print(f"Answering {i+1}/{len(qa_df)}")

    try:
        answer, evidence, used_model = answer_question_text_only(
            question,
            top_k=10
        )

        answer = str(answer).strip()

        if answer == "" or answer.lower() == "nan" or "chatcompletion" in answer.lower():
            answer = "I cannot find enough evidence."

    except Exception as e:
        print("ERROR:", e)
        answer = "I cannot find enough evidence."
        used_model = ""

    results.append({
        "question": question,
        "answer": answer,
        "model_used": used_model
    })

submission_df = pd.DataFrame(results)

cannot_count = submission_df["answer"].astype(str).str.lower().str.contains("cannot find enough evidence").sum()
print("Cannot answers:", cannot_count)

submission_df[["question", "answer", "model_used"]].head(10)

Answering 1/40
Answering 2/40
Answering 3/40
Answering 4/40
Answering 5/40
Answering 6/40
Answering 7/40
Answering 8/40
Answering 9/40
Answering 10/40
Answering 11/40
Answering 12/40
Answering 13/40
Answering 14/40
Answering 15/40
Answering 16/40
Answering 17/40
Answering 18/40
Answering 19/40
Answering 20/40
Answering 21/40
Answering 22/40
Answering 23/40
Answering 24/40
Answering 25/40
Answering 26/40
Answering 27/40
Answering 28/40
Answering 29/40
Answering 30/40
Answering 31/40
Answering 32/40
Answering 33/40
Answering 34/40
Answering 35/40
Answering 36/40
Answering 37/40
Answering 38/40
Answering 39/40
Answering 40/40
Cannot answers: 0


,question,answer,model_used
0,What year where there most casualties from man...,2023,swiss-ai/Apertus-8B-Instruct-2509
1,In what year did the quantity of man-made disa...,2023,swiss-ai/Apertus-8B-Instruct-2509
2,Between 1970 and 2023 what year in the data sh...,2023,swiss-ai/Apertus-8B-Instruct-2509
3,What year between 1994 and 2023 had the most h...,2011,swiss-ai/Apertus-8B-Instruct-2509
4,How much higher are the 2023 insured losses th...,The 2023 insured losses are 21% higher than th...,swiss-ai/Apertus-8B-Instruct-2509
5,Which figure shows the trend in insured losses...,The highest insured loss year on record is 202...,swiss-ai/Apertus-8B-Instruct-2509
6,"Before 2023, what was the highest year on reco...",2011,swiss-ai/Apertus-8B-Instruct-2509
7,What regions does the Swiss Re report on natur...,North America,swiss-ai/Apertus-8B-Instruct-2509
8,What is the highest Benefit to Cost ratio buil...,The highest Benefit to Cost ratio building cod...,swiss-ai/Apertus-8B-Instruct-2509
9,According to the Swiss Re Institute report on ...,The weather events that contributed most to th...,swiss-ai/Apertus-8B-Instruct-2509


In [15]:
submission_df.drop("model_used", axis=1).to_csv("A2Q_submission_1.csv", index=False, sep=";")
print("Saved: A2Q_submission_1.csv")

Saved: A2Q_submission_1.csv
